# Assignment 3: TTC Subway Delay Analysis

**Dataset**: TTC Subway Delay Data (2025) from the [City of Toronto Open Data Portal](https://open.toronto.ca/dataset/ttc-subway-delay-data/)

This notebook loads, cleans, and visualizes TTC subway delay data to uncover:
1. The most common and impactful causes of delays
2. Temporal patterns — when delays are most frequent throughout the week

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import os

In [ ]:
# ── Load data directly from City of Toronto Open Data Portal ────────
DATA_URL = (
    "https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/"
    "996cfe8d-fb35-40ce-b569-698d51fc683b/resource/"
    "0b6e5c52-e993-46d6-8d74-8602ee224457/download/"
    "TTC%20Subway%20Delay%20Data%20since%202025.csv"
)

CODES_URL = (
    "https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/"
    "996cfe8d-fb35-40ce-b569-698d51fc683b/resource/"
    "b2d8f5e0-0997-46b5-8abd-caa685a0290b/download/"
    "Code%20Descriptions.csv"
)

# Read delay records and code descriptions
delays = pd.read_csv(DATA_URL)
codes  = pd.read_csv(CODES_URL)

print(f"Delay records loaded: {len(delays):,}")
print(f"Code descriptions loaded: {len(codes):,}")

In [ ]:
# ── Explore the data ────────────────────────────────────────────────
print("=== Delay Data ===")
display(delays.head())
print(f"\nShape: {delays.shape}")
print(f"\nColumns: {list(delays.columns)}")
print(f"\nMissing values:\n{delays.isnull().sum()}")

In [ ]:
# ── Explore code descriptions ───────────────────────────────────────
print("=== Code Descriptions ===")
display(codes.head(10))
print(f"\nColumns: {list(codes.columns)}")

In [ ]:
# ── Data Cleaning & Preprocessing ──────────────────────────────────

# Standardize column names to lowercase for consistency
delays.columns = delays.columns.str.strip().str.lower().str.replace(' ', '_')
codes.columns  = codes.columns.str.strip().str.lower().str.replace(' ', '_')

# Identify the key columns (column names may vary between years)
print("Delay columns:", list(delays.columns))
print("Code columns:", list(codes.columns))

In [ ]:
# ── Merge delay records with human-readable code descriptions ──────

# Find the code column name in each dataframe
delay_code_col = [c for c in delays.columns if 'code' in c][0]
code_code_col  = [c for c in codes.columns if 'code' in c and 'desc' not in c][0]
code_desc_col  = [c for c in codes.columns if 'desc' in c][0]

print(f"Merging on: delays['{delay_code_col}'] ↔ codes['{code_code_col}']")
print(f"Description column: '{code_desc_col}'")

# Merge to get descriptive names for each delay code
df = delays.merge(
    codes[[code_code_col, code_desc_col]],
    left_on=delay_code_col,
    right_on=code_code_col,
    how='left'
)

print(f"\nMerged dataset shape: {df.shape}")
print(f"Rows with matched descriptions: {df[code_desc_col].notna().sum():,}")

In [ ]:
# ── Identify numeric delay column ──────────────────────────────────
min_delay_col = [c for c in df.columns if 'min' in c and 'delay' in c][0]
print(f"Delay minutes column: '{min_delay_col}'")
print(f"\nDelay stats:\n{df[min_delay_col].describe()}")

In [ ]:
# ── Parse date and time columns ───────────────────────────────────
date_col = [c for c in df.columns if 'date' in c][0]
time_col = [c for c in df.columns if 'time' in c][0]

# Convert date to datetime
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

# Extract day of week and hour
df['day_of_week'] = df[date_col].dt.day_name()
df['hour'] = pd.to_datetime(df[time_col], format='%H:%M', errors='coerce').dt.hour

# If hour parsing failed, try alternative format
if df['hour'].isna().all():
    df['hour'] = pd.to_datetime(df[time_col], errors='coerce').dt.hour

print(f"Day of week values: {df['day_of_week'].unique()}")
print(f"Hour range: {df['hour'].min()} - {df['hour'].max()}")

---
## Visualization 1: Top 10 Causes of TTC Subway Delays

A horizontal bar chart ranking the top 10 delay causes by total accumulated delay minutes.

In [ ]:
# ── Aggregate total delay minutes by cause ─────────────────────────
top_causes = (
    df.groupby(code_desc_col)[min_delay_col]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .sort_values()  # ascending for horizontal bar chart
)

# Shorten long labels for readability (wrap at 40 chars)
labels = [s[:45] + '…' if len(str(s)) > 45 else str(s) for s in top_causes.index]

print("Top 10 delay causes by total minutes:")
for cause, mins in top_causes.items():
    print(f"  {mins:>8,.0f} min — {cause}")

In [ ]:
# ── Visualization 1: Horizontal Bar Chart ──────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

# Use a colorblind-friendly sequential palette
colors = plt.cm.viridis(np.linspace(0.25, 0.85, len(top_causes)))

bars = ax.barh(labels, top_causes.values, color=colors, edgecolor='white', linewidth=0.5)

# Add value labels at the end of each bar
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + top_causes.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'{width:,.0f}',
        va='center', ha='left', fontsize=9, color='#333333'
    )

ax.set_xlabel('Total Delay Minutes', fontsize=12, labelpad=10)
ax.set_title(
    'Top 10 Causes of TTC Subway Delays (2025)',
    fontsize=14, fontweight='bold', pad=15
)

# Format x-axis with thousands separator
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Clean up chart
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=10)

fig.tight_layout()

# Save as high-resolution PNG
output_dir = os.path.dirname(os.path.abspath('__file__'))
fig.savefig(
    os.path.join(output_dir, 'visualization_1_top_delay_causes.png'),
    dpi=300, bbox_inches='tight', facecolor='white'
)
print("Saved: visualization_1_top_delay_causes.png")
plt.show()

---
## Visualization 2: TTC Subway Delay Patterns by Day and Hour

A heatmap revealing when delays are most frequent, broken down by day of the week and hour of the day.  
*(This visualization is also recreated in Google Sheets — the processed pivot data is exported below.)*

In [ ]:
# ── Build pivot table: average delay by day of week × hour ─────────
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

pivot = df.pivot_table(
    values=min_delay_col,
    index='day_of_week',
    columns='hour',
    aggfunc='mean'
).reindex(day_order)

# Fill any missing hour slots with 0
pivot = pivot.fillna(0).round(1)

print(f"Pivot table shape: {pivot.shape}")
display(pivot)

In [ ]:
# ── Visualization 2: Heatmap ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

sns.heatmap(
    pivot,
    cmap='YlOrRd',          # sequential warm palette — colorblind accessible
    annot=True,              # show numeric values in cells
    fmt='.0f',               # no decimal places for readability
    linewidths=0.5,          # grid lines between cells
    linecolor='white',
    cbar_kws={'label': 'Avg Delay (min)', 'shrink': 0.8},
    ax=ax
)

# Format hour labels on x-axis
hour_labels = [f'{int(h)}:00' for h in pivot.columns]
ax.set_xticklabels(hour_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

ax.set_xlabel('Hour of Day', fontsize=12, labelpad=10)
ax.set_ylabel('Day of Week', fontsize=12, labelpad=10)
ax.set_title(
    'Average TTC Subway Delay Duration by Day and Hour (2025)',
    fontsize=14, fontweight='bold', pad=15
)

fig.tight_layout()

# Save as high-resolution PNG
fig.savefig(
    os.path.join(output_dir, 'visualization_2_delay_heatmap.png'),
    dpi=300, bbox_inches='tight', facecolor='white'
)
print("Saved: visualization_2_delay_heatmap.png")
plt.show()

In [ ]:
# ── Export pivot table to CSV for Google Sheets recreation ─────────
export_path = os.path.join(output_dir, 'delay_heatmap_data.csv')
pivot.to_csv(export_path)
print(f"Exported pivot table to: {export_path}")
print("Open this CSV in Google Sheets to recreate the heatmap visualization.")